# 0. 环境恢复
> 断线重连后只需跑这一格，所有东西一次性恢复

In [2]:
# ========================================
# 超级恢复 Cell（断线重连后只跑这一格）
# ========================================
import os, json, re, time
from collections import defaultdict

# 1. 挂载 Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
DRIVE_DIR = "/content/drive/MyDrive/repomind_rag/"

# 2. 装包
import subprocess
subprocess.run(["pip", "install", "-q", "sentence-transformers", "faiss-cpu", "peft", "tiktoken"], check=True)

# 3. 加载 FAISS 索引 + chunks
import faiss
index = faiss.read_index(DRIVE_DIR + "requests_code_v2.index")
with open(DRIVE_DIR + "chunks_metadata_v2.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)
print(f"✅ 索引: {index.ntotal} 向量 | Chunks: {len(chunks)}")

# 4. 加载 bge-m3
from sentence_transformers import SentenceTransformer
embed_model = SentenceTransformer("BAAI/bge-m3")
print("✅ bge-m3 就绪")

# 5. 加载 Reranker
from sentence_transformers import CrossEncoder
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device, max_length=512)
print(f"✅ Reranker 就绪（{device}）")

# 6. 加载评测集
os.system('wget -q "https://raw.githubusercontent.com/yasminn89/tencent-mini/main/project2-repomind/repomind_experiment_data.json" -O /content/experiment_data.json')
with open("/content/experiment_data.json", "r", encoding="utf-8") as f:
    exp_data = json.load(f)
cases = exp_data["evaluation_cases"]
print(f"✅ 评测集: {len(cases)} 条")

# 7. DeepSeek 客户端
from openai import OpenAI
from google.colab import userdata
client = OpenAI(api_key=userdata.get("DEEPSEEK_API_KEY"), base_url="https://api.deepseek.com")
print("✅ DeepSeek client 就绪")

# 8. 检索函数
import numpy as np
def semantic_retrieve(query, k=10):
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(q_emb, k)
    return [(chunks[idx], float(s)) for idx, s in zip(indices[0], scores[0]) if idx >= 0]

def rerank(query, candidates, top_k=5):
    if not candidates:
        return []
    pairs = [[query, c[0].get("text", "")] for c in candidates]
    rerank_scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, rerank_scores), key=lambda x: x[1], reverse=True)
    return [c[0] for c, _ in ranked[:top_k]]

# 9. Prompt 常量
SYSTEM_PROMPT = """你是一名专精于 psf/requests 库源码分析的代码检索专家。

【你熟悉的 requests 核心文件】
- api.py: 用户入口（get/post/put/delete/request 等顶层函数）
- sessions.py: 会话管理（Session类，处理cookies、重定向、配置合并）
- adapters.py: 传输适配器（HTTPAdapter，调用urllib3发送请求）
- models.py: 数据模型（Request/PreparedRequest/Response类）
- auth.py: 认证（HTTPBasicAuth/HTTPDigestAuth/HTTPProxyAuth）
- cookies.py: Cookie处理（RequestsCookieJar 及辅助函数）
- exceptions.py: 异常体系（RequestException 及全部子类）
- hooks.py: 钩子机制（dispatch_hook, default_hooks）
- structures.py: 数据结构（CaseInsensitiveDict, LookupDict）
- utils.py: 工具函数（编码检测、代理处理、URL解析等大量辅助函数）

【架构层次】
用户API层(api.py) → Session层(sessions.py) → Adapter层(adapters.py) → urllib3底层

【回答要求】
1. 明确给出文件名（格式 requests/xxx.py）和函数名或类名
2. 如涉及跨文件调用，列出完整调用链
3. 如 requests 不支持该功能，直接说不支持
4. 简洁直接，不要冗长解释"""

COT_INSTRUCTION = """

请在回答前，先在 <thinking> 标签内完成以下思考步骤：

<thinking>
步骤1 — 查询类型识别
步骤2 — 架构层次定位
步骤3 — 精准定位：具体文件和函数
步骤4 — 验证：定位是否合理
</thinking>

完成思考后，给出简洁明确的最终答案，包含文件名和函数/类名。"""

MAX_CHUNK_CHARS = 2500
def build_rag_prompt(query, retrieved_chunks):
    parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        meta = chunk.get("metadata", {})
        text = chunk.get("text", "")[:MAX_CHUNK_CHARS]
        parts.append(f"【片段{i}】{meta.get('file','?')} → {meta.get('function','?')}\n{text}")
    context = "\n\n".join(parts)
    return f"【检索到的相关代码片段】\n{context}\n\n【用户查询】\n{query}"

# 10. v4 评分函数
def auto_score(response, case):
    r = response.lower()
    expected_file = case.get("expected_file", "").lower()
    expected_func = case.get("expected_function", "").lower()
    if "n/a" in expected_file or expected_file.strip() == "":
        return 2 if any(w in r for w in ["不支持","无法","不能","doesn't support","does not support","no native","no built-in"]) else 0
    file_names = list(set(re.findall(r'(\w+)\.py', expected_file)))
    if not file_names:
        return 0
    file_ok = sum(1 for fn in file_names if fn in r) / len(file_names) >= 0.5
    func_text = expected_func.replace("→"," ").replace(","," ").replace("/"," ").replace("vs"," ")
    func_names = list(set(f for f in re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', func_text)
                          if f.lower() not in {"vs","and","or","the","a","an","request","function","class"} and len(f) >= 3))
    if not func_names:
        return 2 if file_ok else 0
    func_ok = sum(1 for fn in func_names if fn.lower() in r) / len(func_names) >= 0.5
    if file_ok and func_ok:
        return 2
    elif file_ok or func_ok:
        return 1
    return 0

print("\n" + "="*50)
print("✅ 所有变量恢复完毕，可以继续跑消融实验")
print("="*50)

Mounted at /content/drive
✅ 索引: 288 向量 | Chunks: 288


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

✅ Reranker 就绪（cuda）
✅ 评测集: 147 条
✅ DeepSeek client 就绪

✅ 所有变量恢复完毕，可以继续跑消融实验


In [2]:
# ============================================================
# Cell 0（修正版）：恢复 Cell
# ============================================================

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import subprocess, importlib

def install_if_missing(package, import_name=None):
    name = import_name or package
    try:
        importlib.import_module(name)
        print(f"  ✓ {package} 已安装")
    except ImportError:
        print(f"  ↓ 安装 {package}...")
        subprocess.run(["pip", "install", package, "-q"], check=True)

install_if_missing("sentence-transformers", "sentence_transformers")
install_if_missing("faiss-cpu", "faiss")

import json, os, faiss
import numpy as np

DRIVE_DIR = "/content/drive/MyDrive/repomind_rag/"

# 加载 FAISS 索引（v2）
index = faiss.read_index(DRIVE_DIR + "requests_code_v2.index")
print(f"✅ FAISS 索引加载完毕，共 {index.ntotal} 个向量")

# 加载 chunks（v2）
with open(DRIVE_DIR + "chunks_metadata_v2.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)
print(f"✅ Chunks 加载完毕，共 {len(chunks)} 个")

# 看一下 chunk 的字段名，后面要用
print(f"\n【Chunk 字段预览（第0条）】")
print(json.dumps(chunks[0], ensure_ascii=False, indent=2))

Mounted at /content/drive
  ✓ sentence-transformers 已安装
  ✓ faiss-cpu 已安装
✅ FAISS 索引加载完毕，共 288 个向量
✅ Chunks 加载完毕，共 288 个

【Chunk 字段预览（第0条）】
{
  "text": "This is a top-level function named 'request' in file requests/api.py.\n\nSignature: def request(method, url, **kwargs)\n\nDocumentation: Constructs and sends a :class:`Request <Request>`.\n\n:param method: method for the new :class:`Request` object: ``GET``, ``OPTIONS``, ``HEAD``, ``POST``, ``PUT``, ``PATCH``, or ``DELETE``.\n:param url: URL for the new :class:`Request` object.\n:param params: (optional) Dictionary, list of tuples or bytes to send\n    in the query string for the :class:`Request`.\n:param data: (optional) Dictionary, list of tuples, bytes, or file-like\n    object to send in the body of the :class:`Request`.\n:param json: (optional) A JSON serializable Python object to send in the body of the :class:`Request`.\n:param headers: (optional) Dictionary of HTTP Headers to send with the :class:`Request`.\n:param cookies: (op

# 1. RAG 基础组件
加载评测集（147 条）、bge-m3 Embedding 模型、bge-reranker-v2-m3 重排模型

In [3]:
# Cell 1: 下载评测集 + 加载 embedding 模型
import subprocess
subprocess.run(["pip", "install", "sentence-transformers", "-q"])

import json
import os

# 下载评测集
os.system('wget -q "https://raw.githubusercontent.com/yasminn89/tencent-mini/main/project2-repomind/repomind_experiment_data.json" -O /content/experiment_data.json')

with open("/content/experiment_data.json", "r", encoding="utf-8") as f:
    exp_data = json.load(f)

cases = exp_data["evaluation_cases"]
print(f"✅ 评测集加载完毕，共 {len(cases)} 条")
print(f"   第0条预览: {cases[0]}")

# 加载 bge-m3
from sentence_transformers import SentenceTransformer
print("\n⏳ 加载 bge-m3（首次约2-3分钟）...")
embed_model = SentenceTransformer("BAAI/bge-m3")
print("✅ bge-m3 就绪")

✅ 评测集加载完毕，共 147 条
   第0条预览: {'id': 'A01', 'category': 'A_simple', 'difficulty': 1, 'query': 'requests库里发送GET请求的核心函数在哪里？', 'expected_file': 'requests/api.py', 'expected_function': 'get', 'expected_keywords': ['def get', 'params=None', '**kwargs'], 'gold_answer': "requests/api.py 中的 get() 函数，调用 request('GET', url, **kwargs)"}

⏳ 加载 bge-m3（首次约2-3分钟）...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ bge-m3 就绪


In [6]:
# Cell 2: 加载 Reranker
from sentence_transformers import CrossEncoder
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  当前设备: {device}")

print("⏳ 加载 bge-reranker-v2-m3...")
reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3",
    device=device,
    max_length=512
)
print("✅ Reranker 就绪")

🖥️  当前设备: cuda
⏳ 加载 bge-reranker-v2-m3...


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

✅ Reranker 就绪


# 2. 两阶段检索：Reranker 评测
策略：bge-m3 语义粗召回 Top10 → CrossEncoder 精排取 Top5  
评测指标：Recall@1/3/5、MRR

In [ ]:
# Cell 3: 两阶段 Reranker 全量评测
from tqdm import tqdm
import numpy as np

DRIVE_DIR = "/content/drive/MyDrive/repomind_rag/"
FIRST_STAGE_K = 10
FINAL_K = 5

def semantic_retrieve(query, k=FIRST_STAGE_K):
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    scores, indices = index.search(q_emb, k)
    results = []
    for idx, score in zip(indices[0], scores[0]):
        if idx >= 0:
            results.append((chunks[idx], float(score)))
    return results

def rerank(query, candidates, top_k=FINAL_K):
    if not candidates:
        return []
    pairs = [[query, c[0].get("text", "")] for c in candidates]
    rerank_scores = reranker.predict(pairs)
    ranked = sorted(zip(candidates, rerank_scores),
                    key=lambda x: x[1], reverse=True)
    return [c[0] for c, _ in ranked[:top_k]]

def chunk_matches_case(chunk, case):
    expected_file = case.get("expected_file", "").lower()
    expected_func = case.get("expected_function", "").lower()
    meta = chunk.get("metadata", {})
    chunk_file = meta.get("file", "").lower()
    chunk_func = meta.get("function", "").lower()
    file_hit = expected_file and expected_file in chunk_file
    func_hit = expected_func and expected_func == chunk_func
    return file_hit or func_hit

# ---- 开始评测 ----
print(f"🚀 开始评测（{len(cases)} 条，设备: {device}）...")

r1 = r3 = r5 = 0
mrr_sum = 0.0
per_case = []

for case in tqdm(cases, desc="Reranker评测"):
    query = case["query"]

    # Stage 1: 语义粗召回
    candidates = semantic_retrieve(query, k=FIRST_STAGE_K)

    # Stage 2: Reranker 精排
    reranked = rerank(query, candidates, top_k=FINAL_K)

    # 计算命中
    first_hit_rank = None
    for rank, chunk in enumerate(reranked, start=1):
        if chunk_matches_case(chunk, case):
            first_hit_rank = rank
            break

    hit_r1 = 1 if first_hit_rank == 1 else 0
    hit_r3 = 1 if (first_hit_rank and first_hit_rank <= 3) else 0
    hit_r5 = 1 if (first_hit_rank and first_hit_rank <= 5) else 0
    mrr    = 1.0 / first_hit_rank if first_hit_rank else 0.0

    r1 += hit_r1
    r3 += hit_r3
    r5 += hit_r5
    mrr_sum += mrr
    per_case.append({
        "id": case.get("id"),
        "category": case.get("category"),
        "query": query,
        "expected_file": case.get("expected_file"),
        "expected_function": case.get("expected_function"),
        "hit_rank": first_hit_rank,
        "r1": hit_r1, "r3": hit_r3, "r5": hit_r5, "mrr": mrr,
        "top5": [c.get("metadata", {}).get("function", "?") for c in reranked]
    })

n = len(cases)
print("\n" + "="*50)
print("📊 两阶段 Reranker 评测结果")
print("="*50)
print(f"  R@1  = {r1/n*100:.1f}%   (语义基线 45.6%)")
print(f"  R@3  = {r3/n*100:.1f}%   (语义基线 63.9%)")
print(f"  R@5  = {r5/n*100:.1f}%   (语义基线 69.4%)")
print(f"  MRR  = {mrr_sum/n:.3f}   (语义基线 0.554)")
print("="*50)

# 保存结果到 Drive
result = {
    "summary": {
        "R@1": round(r1/n*100, 1),
        "R@3": round(r3/n*100, 1),
        "R@5": round(r5/n*100, 1),
        "MRR": round(mrr_sum/n, 3)
    },
    "details": per_case
}
save_path = DRIVE_DIR + "reranker_eval_results.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)
print(f"\n✅ 结果已保存: {save_path}")

# 3. 端到端 RAG
## 3.1 初版评测（含评分函数 bug）
将检索 Top5 注入 Prompt，调用 DeepSeek 生成答案

In [8]:
# 3. 端到端 RAG
## 3.1 初版评测（含评分函数 bug）
将检索 Top5 注入 Prompt，调用 DeepSeek 生成答案
# Cell 4: 端到端 RAG 评测（初版，存在评分函数 bug，详见 Cell 6 修复）
# 流程: 查询 → 检索Top5 → 注入Prompt → DeepSeek生成 → 评分

from google.colab import userdata
from openai import OpenAI
import time, json

# ---- 初始化 DeepSeek 客户端 ----
client = OpenAI(
    api_key=userdata.get("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

# ---- 从 artifacts 读取 System Prompt（和PE实验保持一致）----
SYSTEM_PROMPT = """你是一名专精于 psf/requests 库源码分析的代码检索专家。

【你熟悉的 requests 核心文件】
- api.py: 用户入口（get/post/put/delete/request 等顶层函数）
- sessions.py: 会话管理（Session类，处理cookies、重定向、配置合并）
- adapters.py: 传输适配器（HTTPAdapter，调用urllib3发送请求）
- models.py: 数据模型（Request/PreparedRequest/Response类）
- auth.py: 认证（HTTPBasicAuth/HTTPDigestAuth/HTTPProxyAuth）
- cookies.py: Cookie处理（RequestsCookieJar 及辅助函数）
- exceptions.py: 异常体系（RequestException 及全部子类）
- hooks.py: 钩子机制（dispatch_hook, default_hooks）
- structures.py: 数据结构（CaseInsensitiveDict, LookupDict）
- utils.py: 工具函数（编码检测、代理处理、URL解析等大量辅助函数）

【架构层次】
用户API层(api.py) → Session层(sessions.py) → Adapter层(adapters.py) → urllib3底层
数据流过: 用户调用 → Request → PreparedRequest → Adapter发送 → Response

【回答要求】
1. 明确给出文件名（格式 requests/xxx.py）和函数名或类名
2. 如涉及跨文件调用，列出完整调用链
3. 如 requests 不支持该功能，直接说requests不支持，不要硬答
4. 简洁直接，不要冗长解释"""

COT_INSTRUCTION = """

请在回答前，先在 <thinking> 标签内完成以下思考步骤：

<thinking>
步骤1 — 查询类型识别：
  - 这是询问"功能在哪实现"，还是"如何使用"，还是"是否支持"？
  - 涉及的核心概念是什么？

步骤2 — 架构层次定位：
  - 这个功能属于 requests 的哪个层次？

步骤3 — 精准定位：
  - 具体在哪个文件的哪个函数或类？
  - 参考上方【检索到的相关代码片段】中的内容

步骤4 — 验证：
  - 我的定位是否合理？
  - 如果 requests 不支持，明确说明，不要硬编。
</thinking>

完成思考后，给出简洁明确的最终答案，包含文件名和函数/类名。"""

# ---- Token 预算控制 ----
MAX_CONTEXT_CHARS = 3000  # 每个chunk截断长度，控制总token数

def build_rag_prompt(query, retrieved_chunks):
    """把检索结果拼成上下文注入Prompt"""
    context_parts = []
    for i, chunk in enumerate(retrieved_chunks, 1):
        meta = chunk.get("metadata", {})
        fname = meta.get("file", "unknown")
        func  = meta.get("function", "unknown")
        text  = chunk.get("text", "")[:MAX_CONTEXT_CHARS]  # 截断控制token
        context_parts.append(f"【片段{i}】{fname} → {func}\n{text}")

    context = "\n\n".join(context_parts)
    user_msg = f"""【检索到的相关代码片段】
{context}

【用户查询】
{query}"""
    return user_msg

# ---- 评分函数（与PE实验v4版保持一致）----
def score_response(response_text, case):
    import re
    expected_file = case.get("expected_file", "").lower()
    expected_func = case.get("expected_function", "").lower()

    # 剥离thinking块
    cleaned = re.sub(r'<thinking>.*?</thinking>', '', response_text,
                     flags=re.DOTALL).lower()

    file_hit = expected_file and expected_file.replace("requests/", "") in cleaned
    func_hit = expected_func and expected_func in cleaned

    if file_hit and func_hit:
        return 2
    elif file_hit or func_hit:
        return 1
    else:
        return 0

# ---- 主评测循环 ----
print(f"🚀 开始端到端 RAG 评测（{len(cases)} 条）...")
print("   预计耗时：约10-15分钟（API限速）\n")

rag_results = []
total_score = 0
errors = 0

for i, case in enumerate(cases):
    query = case["query"]

    try:
        # Step 1: 检索
        candidates = semantic_retrieve(query, k=10)
        reranked   = rerank(query, candidates, top_k=5)

        # Step 2: 构建RAG Prompt
        user_msg = build_rag_prompt(query, reranked)

        # Step 3: 调用 DeepSeek
        resp = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT + COT_INSTRUCTION},
                {"role": "user",   "content": user_msg}
            ],
            max_tokens=1024,
            temperature=0.1
        )
        answer = resp.choices[0].message.content

        # Step 4: 评分
        score = score_response(answer, case)
        total_score += score

        rag_results.append({
            "id": case.get("id"),
            "category": case.get("category"),
            "query": query,
            "expected_file": case.get("expected_file"),
            "expected_function": case.get("expected_function"),
            "score": score,
            "response": answer[:500]  # 只存前500字符省空间
        })

        # 每10条打印进度
        if (i + 1) % 10 == 0:
            current_pct = total_score / ((i+1) * 2) * 100
            print(f"  进度 {i+1}/{len(cases)}，当前综合得分: {current_pct:.1f}%")

        time.sleep(0.5)  # 避免触发API限速

    except Exception as e:
        print(f"  ⚠️  第{i}条出错: {e}")
        errors += 1
        rag_results.append({
            "id": case.get("id"),
            "category": case.get("category"),
            "query": query,
            "score": 0,
            "error": str(e)
        })
        time.sleep(2)
        continue

# ---- 汇总结果 ----
n = len(cases)
final_pct = total_score / (n * 2) * 100

# 按类别统计
from collections import defaultdict
cat_scores = defaultdict(lambda: {"score": 0, "count": 0})
for r in rag_results:
    cat = r.get("category", "unknown")
    cat_scores[cat]["score"] += r.get("score", 0)
    cat_scores[cat]["count"] += 1

print("\n" + "="*55)
print("📊 端到端 RAG 评测结果")
print("="*55)
print(f"  综合得分: {final_pct:.1f}%  (PE-only 基线: 80.3%)")
print(f"  总分: {total_score} / {n*2}")
print(f"  错误条数: {errors}")
print("\n  各类别得分:")
for cat, v in sorted(cat_scores.items()):
    pct = v["score"] / (v["count"] * 2) * 100
    print(f"    {cat:25s}: {pct:.1f}%  ({v['count']}条)")
print("="*55)

# 保存结果
save_path = DRIVE_DIR + "rag_e2e_results.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump({
        "summary": {
            "total_score": total_score,
            "max_score": n * 2,
            "percentage": round(final_pct, 1),
            "pe_only_baseline": 80.3,
            "errors": errors,
            "by_category": {k: round(v["score"]/(v["count"]*2)*100, 1)
                           for k, v in cat_scores.items()}
        },
        "details": rag_results
    }, f, ensure_ascii=False, indent=2)
print(f"\n✅ 结果已保存: {save_path}")

🚀 开始端到端 RAG 评测（147 条）...
   预计耗时：约10-15分钟（API限速）

  进度 10/147，当前综合得分: 95.0%
  进度 20/147，当前综合得分: 70.0%
  进度 30/147，当前综合得分: 60.0%
  进度 40/147，当前综合得分: 63.7%
  进度 50/147，当前综合得分: 59.0%
  进度 60/147，当前综合得分: 65.0%
  进度 70/147，当前综合得分: 70.0%
  进度 80/147，当前综合得分: 73.8%
  进度 90/147，当前综合得分: 76.7%
  进度 100/147，当前综合得分: 78.0%
  进度 110/147，当前综合得分: 80.0%
  进度 120/147，当前综合得分: 80.8%
  进度 130/147，当前综合得分: 79.6%
  进度 140/147，当前综合得分: 81.1%

📊 端到端 RAG 评测结果
  综合得分: 81.3%  (PE-only 基线: 80.3%)
  总分: 239 / 294
  错误条数: 0

  各类别得分:
    A_simple                 : 83.3%  (15条)
    B_cross_file             : 36.7%  (15条)
    C_fuzzy                  : 66.7%  (12条)
    D_edge                   : 43.8%  (8条)
    auto_generated           : 92.8%  (97条)

✅ 结果已保存: /content/drive/MyDrive/repomind_rag/rag_e2e_results.json


## 3.2 评分函数诊断与修复
B_cross_file 类得分异常（96.7% → 36.7%），诊断发现自定义评分函数对多文件期望值（如 `requests/api.py → requests/sessions.py`）的子串匹配失败。  
**修复**：回用低难度 PE 实验的原版 v4 评分函数（按 `.py` 文件名提取 + 至少命中一半）

In [12]:
## 3.2 评分函数诊断与修复
B_cross_file 类得分异常（96.7% → 36.7%），诊断发现自定义评分函数对多文件期望值（如 `requests/api.py → requests/sessions.py`）的子串匹配失败。
**修复**：改用低难度 PE 实验的原版 v4 评分函数（按 `.py` 文件名提取 + 至少命中一半）
# Cell 6（重写）: 用低难度报告中的原版 v4 评分函数重打分
import re, json
from collections import defaultdict

DRIVE_DIR = "/content/drive/MyDrive/repomind_rag/"

def auto_score(response, case):
    """v4 评分函数（与低难度实验完全一致）"""
    r = response.lower()
    expected_file = case.get("expected_file", "").lower()
    expected_func = case.get("expected_function", "").lower()

    # N/A 不支持类
    if "n/a" in expected_file or expected_file.strip() == "":
        not_support_words = ["不支持", "不支援", "无法", "不能",
                             "doesn't support", "does not support",
                             "no native", "no built-in"]
        if any(w in r for w in not_support_words):
            return 2
        return 0

    # 文件名（至少命中一半）
    file_names = list(set(re.findall(r'(\w+)\.py', expected_file)))
    if not file_names:
        return 0
    file_hits = sum(1 for fn in file_names if fn in r)
    file_ok = (file_hits / len(file_names)) >= 0.5

    # 函数名（至少命中一半，保留下划线）
    func_text = expected_func.replace("→"," ").replace(",", " ").replace("/", " ").replace("vs", " ")
    func_candidates = re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', func_text)
    ignore = {"vs", "and", "or", "the", "a", "an", "request", "function", "class"}
    func_names = list(set(f for f in func_candidates
                          if f.lower() not in ignore and len(f) >= 3))

    if not func_names:
        return 2 if file_ok else 0

    func_hits = sum(1 for fn in func_names if fn.lower() in r)
    func_ok = (func_hits / len(func_names)) >= 0.5

    if file_ok and func_ok:
        return 2
    elif file_ok or func_ok:
        return 1
    return 0

# 重打 RAG 结果
with open(DRIVE_DIR + "rag_e2e_results.json", "r", encoding="utf-8") as f:
    rag_data = json.load(f)

cat_old = defaultdict(lambda: {"score": 0, "count": 0})
cat_new = defaultdict(lambda: {"score": 0, "count": 0})
total_old = total_new = 0

for r in rag_data["details"]:
    cat = r.get("category", "unknown")
    old_s = r.get("score", 0)
    new_s = auto_score(r.get("response", ""), {
        "expected_file": r.get("expected_file", ""),
        "expected_function": r.get("expected_function", "")
    })
    cat_old[cat]["score"] += old_s
    cat_old[cat]["count"] += 1
    cat_new[cat]["score"] += new_s
    cat_new[cat]["count"] += 1
    total_old += old_s
    total_new += new_s

n = len(rag_data["details"])
print("="*70)
print("📊 RAG 用原版 v4 评分重打分（基于截断响应，下界估计）")
print("="*70)
print(f"\n{'类别':<25} {'我旧评分':<12} {'原版v4':<12} {'变化'}")
print("-"*70)
for cat in sorted(cat_old.keys()):
    old_pct = cat_old[cat]["score"] / (cat_old[cat]["count"]*2) * 100
    new_pct = cat_new[cat]["score"] / (cat_new[cat]["count"]*2) * 100
    print(f"{cat:<25} {old_pct:>6.1f}%      {new_pct:>6.1f}%     {new_pct-old_pct:+.1f}%")

old_total = total_old / (n*2) * 100
new_total = total_new / (n*2) * 100
print("-"*70)
print(f"{'综合':<25} {old_total:>6.1f}%      {new_total:>6.1f}%     {new_total-old_total:+.1f}%")
print("="*70)
print(f"\n(PE-only 用原版 v4 评分: 80.3%)")
print(f"(若 RAG 提升超 80.3%，说明 RAG 真实有效)")

📊 RAG 用原版 v4 评分重打分（基于截断响应，下界估计）

类别                        我旧评分         原版v4         变化
----------------------------------------------------------------------
A_simple                    83.3%        83.3%     +0.0%
B_cross_file                36.7%       100.0%     +63.3%
C_fuzzy                     66.7%        91.7%     +25.0%
D_edge                      43.8%       100.0%     +56.2%
auto_generated              92.8%        93.3%     +0.5%
----------------------------------------------------------------------
综合                          81.3%        93.2%     +11.9%

(PE-only 用原版 v4 评分: 80.3%)
(若 RAG 提升超 80.3%，说明 RAG 真实有效)


## 3.3 最终版：Token 预算 + 完整评测
- 用原版 v4 评分（与 PE 实验保持完全一致，结果可直接对比）
- 加入 Token 预算控制：每个 chunk 截断 2500 字符，Prompt 总预算 8K tokens
- 保存完整响应（不截断）以便后续分析

In [14]:
!pip install tiktoken -q

In [15]:
# Cell 7: 用原版 v4 重跑 RAG，保存完整响应 + 显式 Token 预算控制
import time, json, tiktoken
from collections import defaultdict
from tqdm import tqdm

DRIVE_DIR = "/content/drive/MyDrive/repomind_rag/"

# ---- Token 预算 ----
# DeepSeek-chat 上下文 128K，但实际控制在 8K 以内保证响应质量
MAX_PROMPT_TOKENS = 8000
MAX_CHUNK_CHARS   = 2500   # 每个 chunk 字符上限
MAX_RESPONSE_TOKENS = 1024

# DeepSeek 用 cl100k_base 编码（兼容 OpenAI tiktoken）
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(enc.encode(text))

def build_rag_prompt_v2(query, retrieved_chunks, budget=MAX_PROMPT_TOKENS):
    """带 Token 预算的 Prompt 构建：超出预算就丢弃后面的 chunks"""
    system_tokens = count_tokens(SYSTEM_PROMPT + COT_INSTRUCTION)
    query_tokens  = count_tokens(query) + 100  # 预留格式开销
    available = budget - system_tokens - query_tokens

    context_parts = []
    used_tokens = 0
    used_chunks = 0

    for chunk in retrieved_chunks:
        meta = chunk.get("metadata", {})
        text = chunk.get("text", "")[:MAX_CHUNK_CHARS]
        part = f"【片段{used_chunks+1}】{meta.get('file','?')} → {meta.get('function','?')}\n{text}"
        part_tokens = count_tokens(part)
        if used_tokens + part_tokens > available:
            break
        context_parts.append(part)
        used_tokens += part_tokens
        used_chunks += 1

    context = "\n\n".join(context_parts)
    return f"""【检索到的相关代码片段】
{context}

【用户查询】
{query}""", used_chunks, used_tokens

# ---- 主循环 ----
print(f"🚀 用原版 v4 评分 + Token 预算控制重跑 RAG（{len(cases)} 条）...\n")

rag_final_results = []
total_score = 0
errors = 0
total_chunks_used = 0
total_context_tokens = 0

for case in tqdm(cases, desc="RAG-final"):
    query = case["query"]
    try:
        # 检索
        candidates = semantic_retrieve(query, k=10)
        reranked   = rerank(query, candidates, top_k=5)

        # 构建 Prompt（带预算）
        user_msg, n_used, ctx_tokens = build_rag_prompt_v2(query, reranked)
        total_chunks_used += n_used
        total_context_tokens += ctx_tokens

        # 调用
        resp = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT + COT_INSTRUCTION},
                {"role": "user",   "content": user_msg}
            ],
            max_tokens=MAX_RESPONSE_TOKENS,
            temperature=0.1
        )
        answer = resp.choices[0].message.content

        # 评分（用原版 v4）
        score = auto_score(answer, case)
        total_score += score

        rag_final_results.append({
            "id": case.get("id"),
            "category": case.get("category"),
            "query": query,
            "expected_file": case.get("expected_file"),
            "expected_function": case.get("expected_function"),
            "score": score,
            "response": answer,  # 完整保存
            "chunks_used": n_used,
            "context_tokens": ctx_tokens
        })
        time.sleep(0.5)

    except Exception as e:
        errors += 1
        rag_final_results.append({
            "id": case.get("id"),
            "category": case.get("category"),
            "query": query,
            "score": 0,
            "error": str(e)
        })
        time.sleep(2)

# ---- 汇总 ----
n = len(cases)
cat_scores = defaultdict(lambda: {"score": 0, "count": 0})
for r in rag_final_results:
    cat = r.get("category", "unknown")
    cat_scores[cat]["score"] += r.get("score", 0)
    cat_scores[cat]["count"] += 1

final_pct = total_score / (n*2) * 100
avg_chunks = total_chunks_used / n
avg_tokens = total_context_tokens / n

print("\n" + "="*65)
print("📊 端到端 RAG 最终结果（原版 v4 评分 + 完整响应）")
print("="*65)
print(f"  综合得分: {final_pct:.1f}%  (PE-only 基线: 80.3%, 提升 +{final_pct-80.3:.1f}%)")
print(f"  错误条数: {errors}")
print(f"  Token预算控制:")
print(f"    平均使用 chunks: {avg_chunks:.1f}/5")
print(f"    平均 context tokens: {avg_tokens:.0f}")
print(f"\n  各类别得分:")
for cat, v in sorted(cat_scores.items()):
    pct = v["score"] / (v["count"]*2) * 100
    print(f"    {cat:25s}: {pct:.1f}%  ({v['count']}条)")
print("="*65)

# 保存
save_path = DRIVE_DIR + "rag_final_results.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump({
        "summary": {
            "percentage": round(final_pct, 1),
            "pe_only_baseline": 80.3,
            "improvement": round(final_pct - 80.3, 1),
            "errors": errors,
            "token_budget": {
                "max_prompt_tokens": MAX_PROMPT_TOKENS,
                "max_chunk_chars": MAX_CHUNK_CHARS,
                "avg_chunks_used": round(avg_chunks, 2),
                "avg_context_tokens": int(avg_tokens)
            },
            "by_category": {k: round(v["score"]/(v["count"]*2)*100, 1)
                           for k, v in cat_scores.items()}
        },
        "details": rag_final_results
    }, f, ensure_ascii=False, indent=2)
print(f"✅ 已保存: {save_path}")

🚀 用原版 v4 评分 + Token 预算控制重跑 RAG（147 条）...



RAG-final: 100%|██████████| 147/147 [11:15<00:00,  4.59s/it]


📊 端到端 RAG 最终结果（原版 v4 评分 + 完整响应）
  综合得分: 93.9%  (PE-only 基线: 80.3%, 提升 +13.6%)
  错误条数: 0
  Token预算控制:
    平均使用 chunks: 5.0/5
    平均 context tokens: 1704

  各类别得分:
    A_simple                 : 86.7%  (15条)
    B_cross_file             : 100.0%  (15条)
    C_fuzzy                  : 87.5%  (12条)
    D_edge                   : 100.0%  (8条)
    auto_generated           : 94.3%  (97条)
✅ 已保存: /content/drive/MyDrive/repomind_rag/rag_final_results.json


In [16]:
# Cell 8: 准备 requests 源码 + AST 切分
import subprocess, os, ast, json

# clone requests（如果还没 clone）
if not os.path.exists("/tmp/requests"):
    subprocess.run(["git", "clone", "https://github.com/psf/requests.git", "/tmp/requests"],
                   check=True)
    print("✅ requests 源码已 clone")
else:
    print("✅ requests 已存在")

# 核心文件列表（和低难度一致）
CORE_FILES = [
    "api.py", "sessions.py", "adapters.py", "models.py", "auth.py",
    "cookies.py", "exceptions.py", "hooks.py", "structures.py", "utils.py"
]
SOURCE_DIR = "/tmp/requests/src/requests/"

# AST 提取所有函数/类
def extract_chunks_from_file(filepath, relative_path):
    with open(filepath, "r", encoding="utf-8") as f:
        source = f.read()
    tree = ast.parse(source)
    chunks = []

    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)):
            docstring = ast.get_docstring(node) or ""
            chunks.append({
                "file": relative_path,
                "name": node.name,
                "type": "class" if isinstance(node, ast.ClassDef) else "function",
                "docstring": docstring,
                "has_docstring": bool(docstring.strip()),
                "line": node.lineno
            })
    return chunks

all_ast_chunks = []
for fname in CORE_FILES:
    fpath = os.path.join(SOURCE_DIR, fname)
    if os.path.exists(fpath):
        chunks = extract_chunks_from_file(fpath, f"requests/{fname}")
        all_ast_chunks.extend(chunks)
        print(f"  {fname}: {len(chunks)} 个 chunks（含 docstring: {sum(c['has_docstring'] for c in chunks)}）")

print(f"\n✅ 总 chunks: {len(all_ast_chunks)}")
print(f"   含 docstring: {sum(c['has_docstring'] for c in all_ast_chunks)}")
print(f"   不含 docstring（之前未利用）: {sum(not c['has_docstring'] for c in all_ast_chunks)}")

✅ requests 源码已 clone
  api.py: 8 个 chunks（含 docstring: 8）
  sessions.py: 31 个 chunks（含 docstring: 24）
  adapters.py: 22 个 chunks（含 docstring: 16）
  models.py: 57 个 chunks（含 docstring: 34）
  auth.py: 28 个 chunks（含 docstring: 8）
  cookies.py: 56 个 chunks（含 docstring: 36）
  exceptions.py: 28 个 chunks（含 docstring: 28）
  hooks.py: 2 个 chunks（含 docstring: 1）
  structures.py: 19 个 chunks（含 docstring: 3）
  utils.py: 47 个 chunks（含 docstring: 39）

✅ 总 chunks: 298
   含 docstring: 197
   不含 docstring（之前未利用）: 101


In [17]:
# Cell 9: AST 扩展 ~ 200 条新查询
import time
from tqdm import tqdm
from openai import OpenAI
from google.colab import userdata

client = OpenAI(
    api_key=userdata.get("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com"
)

# 加载已有的 147 条，提取已用过的 (file, function) 集合做去重
with open("/content/experiment_data.json", "r", encoding="utf-8") as f:
    exp_data = json.load(f)
existing_cases = exp_data["evaluation_cases"]

# 已用过的函数集合
used_funcs = set()
for c in existing_cases:
    f_file = c.get("expected_file", "").lower().strip()
    f_func = c.get("expected_function", "").lower().strip()
    used_funcs.add((f_file, f_func))
print(f"已存在用例覆盖函数数: {len(used_funcs)}")

# ---- 反向生成查询 Prompt ----
def gen_query_for_chunk(chunk):
    """根据 chunk 的 docstring 或签名生成一条自然语言查询"""
    fname = chunk["name"]
    fpath = chunk["file"]
    ftype = chunk["type"]
    docstring = chunk["docstring"][:500] if chunk["has_docstring"] else ""

    if docstring:
        # 有 docstring：基于 docstring 生成
        prompt = f"""根据下面这段代码的 docstring，反向生成一条**自然语言查询**，模拟用户在不知道函数名时如何描述这个功能。

要求：
1. 查询不能直接出现函数名 `{fname}`
2. 长度 10-25 字
3. 像普通开发者那样口语化提问，不要"如何实现 XXX"这种死板句式
4. 只输出查询本身，不要任何解释或引号

函数名（仅你参考，不要出现在查询里）: {fname}
所在文件: {fpath}

Docstring:
{docstring}

查询："""
    else:
        # 无 docstring：基于函数名 + 文件 + 类型推断
        prompt = f"""根据下面这个函数/类的信息（无文档），猜测它的功能并生成一条**自然语言查询**。

要求：
1. 查询不能直接出现函数名 `{fname}`
2. 长度 10-25 字
3. 像普通开发者那样口语化提问
4. 只输出查询本身

函数名（不要出现在查询里）: {fname}
类型: {ftype}
所在文件: {fpath}

查询："""

    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100,
        temperature=0.7  # 多样性
    )
    query = resp.choices[0].message.content.strip()
    # 清理可能的引号
    query = query.strip("\"'「」")
    return query

# ---- 筛选要生成的 chunks（去重）----
chunks_to_generate = []
for chunk in all_ast_chunks:
    key = (chunk["file"].lower(), chunk["name"].lower())
    if key not in used_funcs:
        chunks_to_generate.append(chunk)

print(f"\n待生成的 chunks: {len(chunks_to_generate)} 个")
print(f"  含 docstring: {sum(c['has_docstring'] for c in chunks_to_generate)}")
print(f"  无 docstring: {sum(not c['has_docstring'] for c in chunks_to_generate)}")

# ---- 开始生成 ----
new_cases = []
errors = 0

print("\n🚀 开始生成新查询...")
for i, chunk in enumerate(tqdm(chunks_to_generate, desc="生成")):
    try:
        query = gen_query_for_chunk(chunk)
        if not query or len(query) < 5:
            errors += 1
            continue

        new_cases.append({
            "id": f"EXT_{i:03d}",
            "category": "ext_with_doc" if chunk["has_docstring"] else "ext_no_doc",
            "query": query,
            "expected_file": chunk["file"],
            "expected_function": chunk["name"]
        })
        time.sleep(0.4)

    except Exception as e:
        errors += 1
        time.sleep(2)

print(f"\n✅ 新生成: {len(new_cases)} 条，错误: {errors}")
print(f"\n样本预览（前5条）:")
for c in new_cases[:5]:
    print(f"  [{c['id']}] {c['query']}")
    print(f"        → {c['expected_file']} / {c['expected_function']}")
    print()

# 保存
DRIVE_DIR = "/content/drive/MyDrive/repomind_rag/"
with open(DRIVE_DIR + "ext_cases_ast.json", "w", encoding="utf-8") as f:
    json.dump(new_cases, f, ensure_ascii=False, indent=2)
print(f"✅ 保存到: {DRIVE_DIR}ext_cases_ast.json")

已存在用例覆盖函数数: 129

待生成的 chunks: 183 个
  含 docstring: 88
  无 docstring: 95

🚀 开始生成新查询...


生成: 100%|██████████| 183/183 [04:27<00:00,  1.46s/it]


✅ 新生成: 183 条，错误: 0

样本预览（前5条）:
  [EXT_000] 如何在登录后自动跳转到之前访问的页面？
        → requests/sessions.py / SessionRedirectMixin

  [EXT_001] 如何在Python会话中发送HTTP请求？
        → requests/sessions.py / send

  [EXT_002] 拿到重定向后的链接是什么
        → requests/sessions.py / get_redirect_target

  [EXT_003] 重定向时要不要移除 Authorization 头
        → requests/sessions.py / should_strip_auth

  [EXT_004] 重定向时怎么去掉认证信息避免泄露
        → requests/sessions.py / rebuild_auth

✅ 保存到: /content/drive/MyDrive/repomind_rag/ext_cases_ast.json


In [18]:
# Cell 10: 同义改写扩展 ~ 150 条
import random
random.seed(42)

# 从原 147 + AST 扩展 183 = 330 条里挑 150 条改写
all_existing = existing_cases + new_cases
print(f"总数据池: {len(all_existing)} 条")

# 按类别分层抽样，保证多样性
from collections import defaultdict
by_cat = defaultdict(list)
for c in all_existing:
    by_cat[c.get("category", "unknown")].append(c)

# 各类别抽样比例
SAMPLE_TARGETS = {
    "A_simple": 20,
    "B_cross_file": 20,
    "C_fuzzy": 15,
    "D_edge": 10,
    "auto_generated": 40,
    "ext_with_doc": 25,
    "ext_no_doc": 20
}

cases_to_paraphrase = []
for cat, target in SAMPLE_TARGETS.items():
    pool = by_cat.get(cat, [])
    if pool:
        sampled = random.sample(pool, min(target, len(pool)))
        cases_to_paraphrase.extend(sampled)

print(f"待改写: {len(cases_to_paraphrase)} 条")
for cat, target in SAMPLE_TARGETS.items():
    actual = sum(1 for c in cases_to_paraphrase if c.get("category") == cat)
    print(f"  {cat}: {actual} 条")

# ---- 改写 Prompt ----
def paraphrase_query(original_query):
    """对一条 query 生成 1 条同义改写"""
    prompt = f"""把下面这个关于 Python requests 库的查询，改写成另一种问法，要求：

1. 意思保持完全一致（找的是同一个函数/功能）
2. 用词、句式、口气尽量不同（可以更口语化或更书面化）
3. 不要出现原查询里的关键名词以外的新名词
4. 只输出改写后的查询，不要解释、不要引号

原查询：{original_query}

改写："""
    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=80,
        temperature=0.8
    )
    return resp.choices[0].message.content.strip().strip("\"'「」")

# ---- 开始改写 ----
paraphrased_cases = []
errors = 0

print("\n🚀 开始同义改写...")
for i, case in enumerate(tqdm(cases_to_paraphrase, desc="改写")):
    try:
        new_query = paraphrase_query(case["query"])
        if not new_query or len(new_query) < 5 or new_query == case["query"]:
            errors += 1
            continue

        paraphrased_cases.append({
            "id": f"PARA_{i:03d}",
            "category": f"para_{case.get('category', 'unknown')}",
            "query": new_query,
            "expected_file": case.get("expected_file"),
            "expected_function": case.get("expected_function"),
            "source_id": case.get("id")  # 记录改写来源
        })
        time.sleep(0.3)

    except Exception as e:
        errors += 1
        time.sleep(2)

print(f"\n✅ 改写新增: {len(paraphrased_cases)} 条，错误/重复: {errors}")
print(f"\n改写效果对比样本（前5对）:")
for p in paraphrased_cases[:5]:
    orig = next(c for c in all_existing if c.get("id") == p["source_id"])
    print(f"  原: {orig['query']}")
    print(f"  改: {p['query']}")
    print(f"  → {p['expected_function']}")
    print()

# 保存
with open(DRIVE_DIR + "ext_cases_paraphrase.json", "w", encoding="utf-8") as f:
    json.dump(paraphrased_cases, f, ensure_ascii=False, indent=2)
print(f"✅ 保存到: {DRIVE_DIR}ext_cases_paraphrase.json")

# ---- 汇总最终数据集 ----
final_dataset = all_existing + paraphrased_cases
print(f"\n" + "="*50)
print(f"📊 微调数据集最终统计")
print("="*50)
print(f"  原始 147 条:        {len(existing_cases)}")
print(f"  AST 扩展:           {len(new_cases)}")
print(f"  同义改写:           {len(paraphrased_cases)}")
print(f"  合计:               {len(final_dataset)}")
print("="*50)

总数据池: 330 条
待改写: 135 条
  A_simple: 15 条
  B_cross_file: 15 条
  C_fuzzy: 12 条
  D_edge: 8 条
  auto_generated: 40 条
  ext_with_doc: 25 条
  ext_no_doc: 20 条

🚀 开始同义改写...


改写: 100%|██████████| 135/135 [03:15<00:00,  1.45s/it]


✅ 改写新增: 135 条，错误/重复: 0

改写效果对比样本（前5对）:
  原: URL参数是如何编码和拼接到请求里的？
  改: 在发起请求时，URL里的查询参数是怎么被处理并加进去的？
  → prepare_url

  原: 如何在requests中设置请求超时时间？
  改: 在requests库里怎么配置请求的超时时间
  → request

  原: requests库里发送GET请求的核心函数在哪里？
  改: Python的requests模块中，发起GET请求的主要方法在哪个位置？
  → get

  原: 怎么上传文件（multipart/form-data）？
  改: 如何用multipart/form-data格式把文件发出去？
  → prepare_body

  原: 如何获取响应体的文本内容？
  改: 怎么拿到返回数据里的文字部分
  → text

✅ 保存到: /content/drive/MyDrive/repomind_rag/ext_cases_paraphrase.json

📊 微调数据集最终统计
  原始 147 条:        147
  AST 扩展:           183
  同义改写:           135
  合计:               465


In [19]:
# Cell 10b: 补齐到 500 条（二轮改写）
import random
random.seed(2026)

NEED_MORE = 500 - len(final_dataset)
print(f"还需补充: {NEED_MORE} 条")

# 从 AST 扩展和 auto_generated 里抽（这两类样本多，二轮改写不容易撞）
pool = [c for c in all_existing
        if c.get("category") in ("auto_generated", "ext_with_doc", "ext_no_doc")]
to_paraphrase_v2 = random.sample(pool, NEED_MORE)

def paraphrase_v2(original_query):
    """二轮改写：强制换角度"""
    prompt = f"""把下面这个关于 Python requests 库的查询，**换一种完全不同的角度**重新表达，要求：

1. 找的是同一个函数/功能（答案不能变）
2. 角度要不同：如果原查询是"如何实现X"，改写为"X 在哪里被定义"或"X 的执行流程是什么"；如果原查询偏技术名词，改写为生活化的描述
3. 不要照搬原查询的句式
4. 只输出改写后的查询，不要解释

原查询：{original_query}

改写："""
    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=80,
        temperature=0.9
    )
    return resp.choices[0].message.content.strip().strip("\"'「」")

paraphrased_v2 = []
print(f"\n🚀 开始二轮改写补 {NEED_MORE} 条...")
for i, case in enumerate(tqdm(to_paraphrase_v2, desc="二轮改写")):
    try:
        new_query = paraphrase_v2(case["query"])
        if new_query and len(new_query) >= 5 and new_query != case["query"]:
            paraphrased_v2.append({
                "id": f"PARA2_{i:03d}",
                "category": f"para2_{case.get('category', 'unknown')}",
                "query": new_query,
                "expected_file": case.get("expected_file"),
                "expected_function": case.get("expected_function"),
                "source_id": case.get("id")
            })
        time.sleep(0.3)
    except Exception:
        time.sleep(2)

print(f"\n✅ 二轮改写: {len(paraphrased_v2)} 条")
print(f"\n样本预览:")
for p in paraphrased_v2[:3]:
    orig = next(c for c in all_existing if c.get("id") == p["source_id"])
    print(f"  原: {orig['query']}")
    print(f"  改: {p['query']}")
    print()

# 最终数据集
final_dataset_full = all_existing + paraphrased_cases + paraphrased_v2

# 保存
with open(DRIVE_DIR + "ext_cases_paraphrase_v2.json", "w", encoding="utf-8") as f:
    json.dump(paraphrased_v2, f, ensure_ascii=False, indent=2)

print(f"\n" + "="*50)
print(f"📊 微调数据集最终统计")
print("="*50)
print(f"  原始评测集:         {len(existing_cases):>3} 条")
print(f"  AST 扩展:           {len(new_cases):>3} 条")
print(f"  同义改写（一轮）:   {len(paraphrased_cases):>3} 条")
print(f"  同义改写（二轮）:   {len(paraphrased_v2):>3} 条")
print(f"  ────────────────────────")
print(f"  合计:               {len(final_dataset_full):>3} 条")
print("="*50)

# 保存完整数据集
with open(DRIVE_DIR + "finetune_dataset_raw.json", "w", encoding="utf-8") as f:
    json.dump(final_dataset_full, f, ensure_ascii=False, indent=2)
print(f"\n✅ 完整数据集已保存: {DRIVE_DIR}finetune_dataset_raw.json")

还需补充: 35 条

🚀 开始二轮改写补 35 条...


二轮改写: 100%|██████████| 35/35 [00:58<00:00,  1.66s/it]


✅ 二轮改写: 35 条

样本预览:
  原: 怎么把一个字典转换成更新请求头用的内部序列格式？
  改: 怎样把像{'Content-Type': 'application/json'}这样的配置，变成底层网络请求里一行一行能直接发出去的文本格式？

  原: 进入上下文管理器时执行什么操作
  改: 使用 with 语句时，requests 库在背后悄悄做了哪些准备工作？

  原: 如何初始化一个自定义的请求结构对象？
  改: 除了发送HTTP，有没有类似填快递单的方式，先把一个请求的各个部分（比如地址、内容、格式）组装好，再统一发出？


📊 微调数据集最终统计
  原始评测集:         147 条
  AST 扩展:           183 条
  同义改写（一轮）:   135 条
  同义改写（二轮）:    35 条
  ────────────────────────
  合计:               500 条

✅ 完整数据集已保存: /content/drive/MyDrive/repomind_rag/finetune_dataset_raw.json


In [20]:
# Cell 11: 数据集划分 + LoRA 训练格式转换
import random
import json
from collections import defaultdict

random.seed(42)

# 加载完整数据集
with open(DRIVE_DIR + "finetune_dataset_raw.json", "r", encoding="utf-8") as f:
    full_data = json.load(f)

print(f"加载完整数据集: {len(full_data)} 条")

# ---- 按类别分层抽样：90% train / 10% val ----
by_cat = defaultdict(list)
for c in full_data:
    by_cat[c.get("category", "unknown")].append(c)

train_set, val_set = [], []
for cat, items in by_cat.items():
    random.shuffle(items)
    n_val = max(1, int(round(len(items) * 0.1)))
    val_set.extend(items[:n_val])
    train_set.extend(items[n_val:])

random.shuffle(train_set)
random.shuffle(val_set)

print(f"\n📊 划分结果:")
print(f"  训练集: {len(train_set)} 条")
print(f"  验证集: {len(val_set)} 条")
print(f"\n  各类别分布:")
for cat in sorted(by_cat.keys()):
    train_n = sum(1 for c in train_set if c.get("category") == cat)
    val_n   = sum(1 for c in val_set if c.get("category") == cat)
    print(f"    {cat:25s}: train={train_n:>3}, val={val_n:>2}")

# ---- 转换为 LoRA 训练格式 ----
INSTRUCTION = """你是 psf/requests 库源码检索专家。请根据用户的自然语言查询，定位到 requests 库中具体的文件和函数/类。

输出格式：
- 明确给出文件名（格式 requests/xxx.py）
- 明确给出函数名或类名
- 简洁回答，不要冗长解释"""

def to_finetune_format(case):
    """转换成 alpaca 格式 + 对话格式（两套都生成，备用）"""
    file = case.get("expected_file", "").strip()
    func = case.get("expected_function", "").strip()

    # 标准化 expected_file（去掉箭头，取主文件）
    main_file = file.split("→")[0].strip()
    main_func = func.split("→")[0].strip()

    # 构造高质量答案
    if not main_file or main_file.lower() == "n/a":
        output = f"requests 不支持该功能。"
    elif main_func:
        output = f"在 {main_file} 的 {main_func} 中实现。"
    else:
        output = f"在 {main_file} 中实现。"

    # 如果原数据里有箭头（涉及多文件/调用链），补充说明
    if "→" in file or "→" in func:
        output = f"涉及调用链：{file} → {func}"

    return {
        "instruction": INSTRUCTION,
        "input": case["query"],
        "output": output,
        # 兼容对话格式
        "messages": [
            {"role": "system", "content": INSTRUCTION},
            {"role": "user",   "content": case["query"]},
            {"role": "assistant", "content": output}
        ],
        # 元信息（不参与训练，方便回溯）
        "_meta": {
            "id": case.get("id"),
            "category": case.get("category"),
            "expected_file": file,
            "expected_function": func
        }
    }

train_lora = [to_finetune_format(c) for c in train_set]
val_lora   = [to_finetune_format(c) for c in val_set]

# 保存
with open(DRIVE_DIR + "finetune_train.json", "w", encoding="utf-8") as f:
    json.dump(train_lora, f, ensure_ascii=False, indent=2)
with open(DRIVE_DIR + "finetune_val.json", "w", encoding="utf-8") as f:
    json.dump(val_lora, f, ensure_ascii=False, indent=2)

# 同时存 JSONL 格式（很多训练框架要 jsonl）
with open(DRIVE_DIR + "finetune_train.jsonl", "w", encoding="utf-8") as f:
    for item in train_lora:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")
with open(DRIVE_DIR + "finetune_val.jsonl", "w", encoding="utf-8") as f:
    for item in val_lora:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print(f"\n✅ LoRA 训练数据已保存:")
print(f"   {DRIVE_DIR}finetune_train.json / .jsonl  ({len(train_lora)} 条)")
print(f"   {DRIVE_DIR}finetune_val.json / .jsonl    ({len(val_lora)} 条)")

# ---- 样本预览 ----
print(f"\n📋 训练样本预览（前 3 条）:")
print("="*70)
for item in train_lora[:3]:
    print(f"  [{item['_meta']['id']} | {item['_meta']['category']}]")
    print(f"  Input : {item['input']}")
    print(f"  Output: {item['output']}")
    print("-"*70)

加载完整数据集: 500 条

📊 划分结果:
  训练集: 447 条
  验证集: 53 条

  各类别分布:
    A_simple                 : train= 13, val= 2
    B_cross_file             : train= 13, val= 2
    C_fuzzy                  : train= 11, val= 1
    D_edge                   : train=  7, val= 1
    auto_generated           : train= 87, val=10
    ext_no_doc               : train= 85, val=10
    ext_with_doc             : train= 79, val= 9
    para2_auto_generated     : train= 10, val= 1
    para2_ext_no_doc         : train=  7, val= 1
    para2_ext_with_doc       : train= 14, val= 2
    para_A_simple            : train= 13, val= 2
    para_B_cross_file        : train= 13, val= 2
    para_C_fuzzy             : train= 11, val= 1
    para_D_edge              : train=  7, val= 1
    para_auto_generated      : train= 36, val= 4
    para_ext_no_doc          : train= 18, val= 2
    para_ext_with_doc        : train= 23, val= 2

✅ LoRA 训练数据已保存:
   /content/drive/MyDrive/repomind_rag/finetune_train.json / .jsonl  (447 条)
   /content/dr

In [25]:
# Cell 12: 清理 transformers 残留，重装
!pip uninstall -y transformers tokenizers
!pip install -q transformers==4.46.3 peft==0.13.0 bitsandbytes==0.43.3 accelerate

print("\n" + "="*60)
print("重启会话才能生效！")
print("="*60)

print("")
print("重启完成后按顺序跑：")
print("  Cell 0  → 挂 Drive + 加载 index/chunks")
print("  Cell 1  → 加载评测集 + bge-m3")
print("  Cell 13 → 加载 Qwen（这次应该不会报错）")
print("="*60)

Found existing installation: transformers 4.44.2
Uninstalling transformers-4.44.2:
  Successfully uninstalled transformers-4.44.2
Found existing installation: tokenizers 0.19.1
Uninstalling tokenizers-0.19.1:
  Successfully uninstalled tokenizers-0.19.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 84.8 MB/s eta 0:00:00

⚠️  必须重启会话才能生效！
操作：菜单栏 → 运行时 → 重新启动会话
（不是 '重启运行时'，是 '重新启动会话'）

重启完成后按顺序跑：
  Cell 0  → 挂 Drive + 加载 index/chunks
  Cell 1  → 加载评测集 + bge-m3
  Cell 13 → 加载 Qwen（这次应该不会报错）


In [4]:
# Cell 13（fp16版）: 加载 Qwen2.5-1.5B + LoRA（跳过 bitsandbytes）
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

print("⏳ 加载 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("⏳ 加载基座模型（fp16，约 1-2 分钟）...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)
model.config.use_cache = False

# ---- LoRA 配置 ----
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

trainable, total = 0, 0
for n, p in model.named_parameters():
    total += p.numel()
    if p.requires_grad:
        trainable += p.numel()

print(f"\n✅ LoRA 配置完成")
print(f"   可训练参数: {trainable/1e6:.2f}M / 总参数: {total/1e6:.0f}M")
print(f"   可训练占比: {trainable/total*100:.2f}%")
print(f"   GPU 显存占用: {torch.cuda.memory_allocated()/1e9:.2f} GB")

⏳ 加载 tokenizer...
⏳ 加载基座模型（fp16，约 1-2 分钟）...

✅ LoRA 配置完成
   可训练参数: 18.46M / 总参数: 1562M
   可训练占比: 1.18%
   GPU 显存占用: 5.43 GB


In [6]:
# 升级 bitsandbytes（修复 triton.ops 问题）
!pip install -q -U bitsandbytes

import bitsandbytes as bnb
print(f"✅ bitsandbytes 版本: {bnb.__version__}")
print("\n⚠️  必须重启会话才生效")
print("菜单 → 运行时 → 重新启动会话")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.2 MB/s eta 0:00:00


ImportError: cannot import name 'has_avx512bf16' from 'bitsandbytes.functional' (/usr/local/lib/python3.12/dist-packages/bitsandbytes/functional.py)

In [1]:
# 验证 Cell：确认环境正常
import bitsandbytes as bnb
import torch
print(f"✅ bitsandbytes: {bnb.__version__}")
print(f"✅ CUDA: {torch.cuda.is_available()}")
print(f"✅ GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A'}")

✅ bitsandbytes: 0.49.2
✅ CUDA: True
✅ GPU: Tesla T4


In [5]:
# Cell 14: 数据加载 + tokenize
import json
from datasets import Dataset

with open(DRIVE_DIR + "finetune_train.json", "r", encoding="utf-8") as f:
    train_raw = json.load(f)
with open(DRIVE_DIR + "finetune_val.json", "r", encoding="utf-8") as f:
    val_raw = json.load(f)

print(f"训练集: {len(train_raw)} 条")
print(f"验证集: {len(val_raw)} 条")

# Qwen 的 chat template
def format_chat(example):
    """用 Qwen 的 chat template 格式化"""
    messages = example["messages"]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

def tokenize_fn(example):
    encoded = tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding=False
    )
    # Causal LM: labels = input_ids
    encoded["labels"] = encoded["input_ids"].copy()
    return encoded

train_ds = Dataset.from_list(train_raw).map(format_chat).map(tokenize_fn, remove_columns=["text"])
val_ds   = Dataset.from_list(val_raw).map(format_chat).map(tokenize_fn, remove_columns=["text"])

# 移除 model 不要的列
keep_cols = ["input_ids", "attention_mask", "labels"]
train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep_cols])
val_ds   = val_ds.remove_columns([c for c in val_ds.column_names if c not in keep_cols])

print(f"\n✅ Tokenize 完成")
print(f"\n样本预览（解码后）:")
sample_ids = train_ds[0]["input_ids"]
print(tokenizer.decode(sample_ids))

训练集: 447 条
验证集: 53 条


Map:   0%|          | 0/447 [00:00<?, ? examples/s]

Map:   0%|          | 0/447 [00:00<?, ? examples/s]

Map:   0%|          | 0/53 [00:00<?, ? examples/s]

Map:   0%|          | 0/53 [00:00<?, ? examples/s]


✅ Tokenize 完成

样本预览（解码后）:
<|im_start|>system
你是 psf/requests 库源码检索专家。请根据用户的自然语言查询，定位到 requests 库中具体的文件和函数/类。

输出格式：
- 明确给出文件名（格式 requests/xxx.py）
- 明确给出函数名或类名
- 简洁回答，不要冗长解释<|im_end|>
<|im_start|>user
如何把Morsel对象转换成包含单一键值对的Cookie？<|im_end|>
<|im_start|>assistant
在 requests/cookies.py 的 morsel_to_cookie 中实现。<|im_end|>



In [8]:
# Cell 15: LoRA 训练 —— 用自定义 collator 同时 pad labels
from transformers import TrainingArguments, Trainer
import torch
import os

OUTPUT_DIR = "/content/drive/MyDrive/repomind_rag/lora_qwen2_5_1_5b"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---- 自定义 collator：同时对 input_ids / attention_mask / labels 做 padding ----
class CausalLMCollator:
    def __init__(self, tokenizer, pad_to_multiple_of=8):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of
        self.pad_token_id = tokenizer.pad_token_id

    def __call__(self, features):
        # 找到 batch 内最长长度，向上对齐到 8 的倍数（fp16 训练效率）
        max_len = max(len(f["input_ids"]) for f in features)
        if self.pad_to_multiple_of:
            max_len = ((max_len + self.pad_to_multiple_of - 1)
                       // self.pad_to_multiple_of) * self.pad_to_multiple_of

        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for f in features:
            ids = f["input_ids"]
            mask = f["attention_mask"]
            labels = f["labels"]
            pad_len = max_len - len(ids)

            batch["input_ids"].append(ids + [self.pad_token_id] * pad_len)
            batch["attention_mask"].append(mask + [0] * pad_len)
            # labels 填 -100 让 loss 忽略 padding 位置
            batch["labels"].append(labels + [-100] * pad_len)

        return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    optim="adamw_torch",
    report_to="none",
    seed=42,
    remove_unused_columns=False  # 防止 trainer 把我们要的列删掉
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=CausalLMCollator(tokenizer),
    tokenizer=tokenizer
)

print("🚀 开始 LoRA 训练（预计 20-40 分钟）...")
print(f"   总 steps: 约 {len(train_ds) // 16 * 3} 步\n")

train_result = trainer.train()

# 保存
model.save_pretrained(OUTPUT_DIR + "/final")
tokenizer.save_pretrained(OUTPUT_DIR + "/final")
print(f"\n✅ 训练完成: {OUTPUT_DIR}/final")

import pandas as pd
log_df = pd.DataFrame(trainer.state.log_history)
print("\n📊 训练历史:")
print(log_df[["epoch", "loss", "eval_loss"]].dropna(how="all").to_string())

/tmp/ipykernel_40853/2881169663.py:60: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


🚀 开始 LoRA 训练（预计 20-40 分钟）...
   总 steps: 约 81 步



Epoch,Training Loss,Validation Loss
1,0.710100,0.451048
2,0.395100,0.410480
3,0.335000,0.405283



✅ 训练完成: /content/drive/MyDrive/repomind_rag/lora_qwen2_5_1_5b/final

📊 训练历史:
       epoch    loss  eval_loss
0   0.357143  2.9172        NaN
1   0.714286  0.7101        NaN
2   1.000000     NaN   0.451048
3   1.071429  0.4731        NaN
4   1.428571  0.3913        NaN
5   1.785714  0.3951        NaN
6   2.000000     NaN   0.410480
7   2.142857  0.3490        NaN
8   2.500000  0.3352        NaN
9   2.857143  0.3350        NaN
10  3.000000     NaN   0.405283
11  3.000000     NaN        NaN


In [9]:
# Cell 16: 微调模型评测（Fine-tune only）
import torch
import re, json
from tqdm import tqdm
from collections import defaultdict

DRIVE_DIR = "/content/drive/MyDrive/repomind_rag/"

# ---- v4 评分函数（与 PE 实验完全一致）----
def auto_score(response, case):
    r = response.lower()
    expected_file = case.get("expected_file", "").lower()
    expected_func = case.get("expected_function", "").lower()

    if "n/a" in expected_file or expected_file.strip() == "":
        not_support_words = ["不支持", "不支援", "无法", "不能",
                             "doesn't support", "does not support",
                             "no native", "no built-in"]
        return 2 if any(w in r for w in not_support_words) else 0

    file_names = list(set(re.findall(r'(\w+)\.py', expected_file)))
    if not file_names:
        return 0
    file_ok = sum(1 for fn in file_names if fn in r) / len(file_names) >= 0.5

    func_text = expected_func.replace("→"," ").replace(",", " ").replace("/", " ").replace("vs", " ")
    func_candidates = re.findall(r'\b[a-zA-Z_][a-zA-Z0-9_]*\b', func_text)
    ignore = {"vs", "and", "or", "the", "a", "an", "request", "function", "class"}
    func_names = list(set(f for f in func_candidates
                          if f.lower() not in ignore and len(f) >= 3))

    if not func_names:
        return 2 if file_ok else 0
    func_ok = sum(1 for fn in func_names if fn.lower() in r) / len(func_names) >= 0.5

    if file_ok and func_ok:
        return 2
    elif file_ok or func_ok:
        return 1
    return 0

# ---- 推理 ----
INSTRUCTION = """你是 psf/requests 库源码检索专家。请根据用户的自然语言查询，定位到 requests 库中具体的文件和函数/类。

输出格式：
- 明确给出文件名（格式 requests/xxx.py）
- 明确给出函数名或类名
- 简洁回答，不要冗长解释"""

model.eval()

def generate(query, max_new_tokens=150):
    messages = [
        {"role": "system", "content": INSTRUCTION},
        {"role": "user", "content": query}
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,           # 贪心解码，可复现
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

# ---- 评测 ----
print(f"🚀 微调模型评测（{len(cases)} 条）...")

ft_results = []
total_score = 0

for case in tqdm(cases, desc="FT评测"):
    try:
        answer = generate(case["query"])
        score = auto_score(answer, case)
        total_score += score
        ft_results.append({
            "id": case.get("id"),
            "category": case.get("category"),
            "query": case["query"],
            "expected_file": case.get("expected_file"),
            "expected_function": case.get("expected_function"),
            "score": score,
            "response": answer
        })
    except Exception as e:
        ft_results.append({
            "id": case.get("id"),
            "category": case.get("category"),
            "query": case["query"],
            "score": 0,
            "error": str(e)
        })

# ---- 汇总 ----
n = len(cases)
cat_scores = defaultdict(lambda: {"score": 0, "count": 0})
for r in ft_results:
    cat = r.get("category", "unknown")
    cat_scores[cat]["score"] += r.get("score", 0)
    cat_scores[cat]["count"] += 1

final_pct = total_score / (n*2) * 100
print("\n" + "="*60)
print("📊 Fine-tune only 评测结果")
print("="*60)
print(f"  综合得分: {final_pct:.1f}%")
print(f"  (基线: 68.7% | PE-only: 80.3% | RAG+PE: 93.9%)")
print(f"\n  各类别得分:")
for cat, v in sorted(cat_scores.items()):
    pct = v["score"] / (v["count"]*2) * 100
    print(f"    {cat:25s}: {pct:.1f}%  ({v['count']}条)")
print("="*60)

# 保存
save_path = DRIVE_DIR + "finetune_only_results.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump({
        "summary": {
            "percentage": round(final_pct, 1),
            "by_category": {k: round(v["score"]/(v["count"]*2)*100, 1)
                           for k, v in cat_scores.items()}
        },
        "details": ft_results
    }, f, ensure_ascii=False, indent=2)
print(f"✅ 已保存: {save_path}")

🚀 微调模型评测（147 条）...


FT评测:   0%|          | 0/147 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
FT评测: 100%|██████████| 147/147 [02:24<00:00,  1.01it/s]


📊 Fine-tune only 评测结果
  综合得分: 67.0%
  (基线: 68.7% | PE-only: 80.3% | RAG+PE: 93.9%)

  各类别得分:
    A_simple                 : 83.3%  (15条)
    B_cross_file             : 76.7%  (15条)
    C_fuzzy                  : 75.0%  (12条)
    D_edge                   : 50.0%  (8条)
    auto_generated           : 63.4%  (97条)
✅ 已保存: /content/drive/MyDrive/repomind_rag/finetune_only_results.json


In [10]:
# Cell 16b: Qwen2.5-1.5B 微调前的 baseline 评测（关掉 LoRA adapter）
import torch
from tqdm import tqdm
from collections import defaultdict

# 临时禁用 LoRA adapter，只用原始 1.5B 模型推理
with model.disable_adapter():
    print("✅ 已禁用 LoRA，当前为 1.5B 原始模型")

    print(f"\n🚀 1.5B 基线评测（{len(cases)} 条）...")
    base_results = []
    total_score = 0

    model.eval()
    for case in tqdm(cases, desc="1.5B基线"):
        try:
            with torch.no_grad():
                answer = generate(case["query"])
            score = auto_score(answer, case)
            total_score += score
            base_results.append({
                "id": case.get("id"),
                "category": case.get("category"),
                "query": case["query"],
                "expected_file": case.get("expected_file"),
                "expected_function": case.get("expected_function"),
                "score": score,
                "response": answer
            })
        except Exception as e:
            base_results.append({
                "id": case.get("id"),
                "score": 0, "error": str(e)
            })

# 汇总
n = len(cases)
cat_scores = defaultdict(lambda: {"score": 0, "count": 0})
for r in base_results:
    cat = r.get("category", "unknown")
    cat_scores[cat]["score"] += r.get("score", 0)
    cat_scores[cat]["count"] += 1

final_pct = total_score / (n*2) * 100
print("\n" + "="*60)
print("📊 Qwen2.5-1.5B 微调前 baseline")
print("="*60)
print(f"  综合得分: {final_pct:.1f}%")
print(f"  vs Fine-tune: 67.0%")
print(f"\n  各类别得分:")
for cat, v in sorted(cat_scores.items()):
    pct = v["score"] / (v["count"]*2) * 100
    print(f"    {cat:25s}: {pct:.1f}%  ({v['count']}条)")
print("="*60)

# 保存
save_path = DRIVE_DIR + "qwen_baseline_results.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump({
        "summary": {
            "percentage": round(final_pct, 1),
            "by_category": {k: round(v["score"]/(v["count"]*2)*100, 1)
                           for k, v in cat_scores.items()}
        },
        "details": base_results
    }, f, ensure_ascii=False, indent=2)
print(f"✅ 已保存: {save_path}")

✅ 已禁用 LoRA，当前为 1.5B 原始模型

🚀 1.5B 基线评测（147 条）...


1.5B基线: 100%|██████████| 147/147 [11:15<00:00,  4.59s/it]


📊 Qwen2.5-1.5B 微调前 baseline
  综合得分: 26.2%
  vs Fine-tune: 67.0%

  各类别得分:
    A_simple                 : 46.7%  (15条)
    B_cross_file             : 40.0%  (15条)
    C_fuzzy                  : 20.8%  (12条)
    D_edge                   : 31.2%  (8条)
    auto_generated           : 21.1%  (97条)
✅ 已保存: /content/drive/MyDrive/repomind_rag/qwen_baseline_results.json


In [11]:
# 状态检查
import sys
print("✅ 已加载" if "semantic_retrieve" in dir() else "❌ semantic_retrieve 丢失")
print("✅ 已加载" if "rerank" in dir() else "❌ rerank 丢失")
print("✅ 已加载" if "client" in dir() else "❌ client (DeepSeek) 丢失")
print("✅ 已加载" if "embed_model" in dir() else "❌ embed_model 丢失")
print("✅ 已加载" if "reranker" in dir() else "❌ reranker 丢失")
print("✅ 已加载" if "auto_score" in dir() else "❌ auto_score 丢失")

❌ semantic_retrieve 丢失
❌ rerank 丢失
❌ client (DeepSeek) 丢失
✅ 已加载
❌ reranker 丢失
✅ 已加载


In [3]:
# Cell A: RAG only（DeepSeek，无 PE Prompt）
import time
from tqdm import tqdm
from collections import defaultdict

RAG_ONLY_PROMPT = "请根据以下代码片段，回答用户的查询，给出具体的文件名和函数/类名。"

print(f"🚀 RAG only 评测（{len(cases)} 条，无 PE Prompt）...")

rag_only_results = []
total_score = 0

for case in tqdm(cases, desc="RAG-only"):
    query = case["query"]
    try:
        candidates = semantic_retrieve(query, k=10)
        reranked = rerank(query, candidates, top_k=5)
        user_msg = build_rag_prompt(query, reranked)

        resp = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": RAG_ONLY_PROMPT},
                {"role": "user", "content": user_msg}
            ],
            max_tokens=512,
            temperature=0.1
        )
        answer = resp.choices[0].message.content
        score = auto_score(answer, case)
        total_score += score

        rag_only_results.append({
            "id": case.get("id"),
            "category": case.get("category"),
            "query": query,
            "expected_file": case.get("expected_file"),
            "expected_function": case.get("expected_function"),
            "score": score,
            "response": answer
        })
        time.sleep(0.5)

    except Exception as e:
        rag_only_results.append({
            "id": case.get("id"),
            "category": case.get("category"),
            "query": query,
            "score": 0, "error": str(e)
        })
        time.sleep(2)

n = len(cases)
cat_scores = defaultdict(lambda: {"score": 0, "count": 0})
for r in rag_only_results:
    cat = r.get("category", "unknown")
    cat_scores[cat]["score"] += r.get("score", 0)
    cat_scores[cat]["count"] += 1

final_pct = total_score / (n*2) * 100
print("\n" + "="*60)
print("📊 RAG only 评测结果")
print("="*60)
print(f"  综合得分: {final_pct:.1f}%")
print(f"  (基线: 68.7% | PE only: 80.3% | PE+RAG: 93.9%)")
print(f"\n  各类别得分:")
for cat, v in sorted(cat_scores.items()):
    pct = v["score"] / (v["count"]*2) * 100
    print(f"    {cat:25s}: {pct:.1f}%  ({v['count']}条)")
print("="*60)

DRIVE_DIR = "/content/drive/MyDrive/repomind_rag/"
with open(DRIVE_DIR + "rag_only_results.json", "w", encoding="utf-8") as f:
    json.dump({
        "summary": {
            "percentage": round(final_pct, 1),
            "by_category": {k: round(v["score"]/(v["count"]*2)*100, 1)
                           for k, v in cat_scores.items()}
        },
        "details": rag_only_results
    }, f, ensure_ascii=False, indent=2)
print(f"✅ 已保存: {DRIVE_DIR}rag_only_results.json")

🚀 RAG only 评测（147 条，无 PE Prompt）...


RAG-only: 100%|██████████| 147/147 [14:25<00:00,  5.89s/it]


📊 RAG only 评测结果
  综合得分: 94.2%
  (基线: 68.7% | PE only: 80.3% | PE+RAG: 93.9%)

  各类别得分:
    A_simple                 : 90.0%  (15条)
    B_cross_file             : 100.0%  (15条)
    C_fuzzy                  : 83.3%  (12条)
    D_edge                   : 81.2%  (8条)
    auto_generated           : 96.4%  (97条)
✅ 已保存: /content/drive/MyDrive/repomind_rag/rag_only_results.json


In [5]:
!pip install -q torchao --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 33.9 MB/s eta 0:00:00


In [6]:
# Cell B0: 加载微调后的 Qwen2.5-1.5B + LoRA adapter
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

DRIVE_DIR = "/content/drive/MyDrive/repomind_rag/"
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
LORA_PATH = DRIVE_DIR + "lora_qwen2_5_1_5b/final"

print("⏳ 加载 tokenizer...")
tokenizer_ft = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer_ft.pad_token is None:
    tokenizer_ft.pad_token = tokenizer_ft.eos_token

print("⏳ 加载基座模型（fp16）...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

print("⏳ 加载 LoRA adapter...")
ft_model = PeftModel.from_pretrained(base_model, LORA_PATH)
ft_model.eval()
print("✅ 微调模型加载完毕")

# 推理函数
FT_INSTRUCTION = """你是 psf/requests 库源码检索专家。请根据用户的自然语言查询，定位到 requests 库中具体的文件和函数/类。

输出格式：
- 明确给出文件名（格式 requests/xxx.py）
- 明确给出函数名或类名
- 简洁回答，不要冗长解释"""

def ft_generate(query, use_system_prompt=True, context=None):
    """微调模型推理
    use_system_prompt: 是否用完整 System Prompt（False = 裸推理）
    context: RAG 检索结果字符串（None = 不用 RAG）
    """
    sys = FT_INSTRUCTION if use_system_prompt else "你是一个代码助手。"
    if context:
        user_content = f"{context}\n\n【用户查询】\n{query}"
    else:
        user_content = query

    messages = [
        {"role": "system", "content": sys},
        {"role": "user", "content": user_content}
    ]
    prompt = tokenizer_ft.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer_ft(prompt, return_tensors="pt").to(ft_model.device)

    with torch.no_grad():
        outputs = ft_model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            pad_token_id=tokenizer_ft.pad_token_id,
            eos_token_id=tokenizer_ft.eos_token_id
        )
    response = tokenizer_ft.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )
    return response.strip()

print("✅ ft_generate 函数就绪")

⏳ 加载 tokenizer...
⏳ 加载基座模型（fp16）...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

⏳ 加载 LoRA adapter...


✅ 微调模型加载完毕
✅ ft_generate 函数就绪


In [7]:
# Cell B: Qwen-1.5B + PE only（有 System Prompt，无 RAG）
from tqdm import tqdm
from collections import defaultdict

print(f"🚀 PE + Fine-tune 评测（{len(cases)} 条）...")

pe_ft_results = []
total_score = 0

for case in tqdm(cases, desc="PE+FT"):
    try:
        answer = ft_generate(case["query"], use_system_prompt=True, context=None)
        score = auto_score(answer, case)
        total_score += score
        pe_ft_results.append({
            "id": case.get("id"), "category": case.get("category"),
            "query": case["query"],
            "expected_file": case.get("expected_file"),
            "expected_function": case.get("expected_function"),
            "score": score, "response": answer
        })
    except Exception as e:
        pe_ft_results.append({"id": case.get("id"), "category": case.get("category"),
                               "query": case["query"], "score": 0, "error": str(e)})

n = len(cases)
cat_scores = defaultdict(lambda: {"score": 0, "count": 0})
for r in pe_ft_results:
    cat = r.get("category","unknown")
    cat_scores[cat]["score"] += r.get("score", 0)
    cat_scores[cat]["count"] += 1

final_pct = total_score / (n*2) * 100
print("\n" + "="*55)
print("📊 PE + Fine-tune 评测结果")
print("="*55)
print(f"  综合得分: {final_pct:.1f}%  (FT only: 67.0%)")
for cat, v in sorted(cat_scores.items()):
    print(f"    {cat:25s}: {v['score']/(v['count']*2)*100:.1f}%  ({v['count']}条)")
print("="*55)

import json
DRIVE_DIR = "/content/drive/MyDrive/repomind_rag/"
with open(DRIVE_DIR + "pe_ft_results.json", "w", encoding="utf-8") as f:
    json.dump({"summary": {"percentage": round(final_pct,1),
               "by_category": {k: round(v["score"]/(v["count"]*2)*100,1) for k,v in cat_scores.items()}},
               "details": pe_ft_results}, f, ensure_ascii=False, indent=2)
print(f"✅ 已保存")

🚀 PE + Fine-tune 评测（147 条）...


PE+FT: 100%|██████████| 147/147 [03:39<00:00,  1.50s/it]


📊 PE + Fine-tune 评测结果
  综合得分: 66.7%  (FT only: 67.0%)
    A_simple                 : 83.3%  (15条)
    B_cross_file             : 76.7%  (15条)
    C_fuzzy                  : 75.0%  (12条)
    D_edge                   : 50.0%  (8条)
    auto_generated           : 62.9%  (97条)
✅ 已保存


In [8]:
# Cell C: Qwen-1.5B + RAG（无 PE System Prompt）
from tqdm import tqdm
from collections import defaultdict
import time

print(f"🚀 RAG + Fine-tune 评测（{len(cases)} 条）...")

rag_ft_results = []
total_score = 0

for case in tqdm(cases, desc="RAG+FT"):
    try:
        candidates = semantic_retrieve(case["query"], k=10)
        reranked = rerank(case["query"], candidates, top_k=5)
        context = build_rag_prompt(case["query"], reranked)
        answer = ft_generate(case["query"], use_system_prompt=False, context=context)
        score = auto_score(answer, case)
        total_score += score
        rag_ft_results.append({
            "id": case.get("id"), "category": case.get("category"),
            "query": case["query"],
            "expected_file": case.get("expected_file"),
            "expected_function": case.get("expected_function"),
            "score": score, "response": answer
        })
    except Exception as e:
        rag_ft_results.append({"id": case.get("id"), "category": case.get("category"),
                                "query": case["query"], "score": 0, "error": str(e)})

n = len(cases)
cat_scores = defaultdict(lambda: {"score": 0, "count": 0})
for r in rag_ft_results:
    cat = r.get("category","unknown")
    cat_scores[cat]["score"] += r.get("score", 0)
    cat_scores[cat]["count"] += 1

final_pct = total_score / (n*2) * 100
print("\n" + "="*55)
print("📊 RAG + Fine-tune 评测结果")
print("="*55)
print(f"  综合得分: {final_pct:.1f}%  (FT only: 67.0%)")
for cat, v in sorted(cat_scores.items()):
    print(f"    {cat:25s}: {v['score']/(v['count']*2)*100:.1f}%  ({v['count']}条)")
print("="*55)

with open(DRIVE_DIR + "rag_ft_results.json", "w", encoding="utf-8") as f:
    json.dump({"summary": {"percentage": round(final_pct,1),
               "by_category": {k: round(v["score"]/(v["count"]*2)*100,1) for k,v in cat_scores.items()}},
               "details": rag_ft_results}, f, ensure_ascii=False, indent=2)
print(f"✅ 已保存")

🚀 RAG + Fine-tune 评测（147 条）...


RAG+FT: 100%|██████████| 147/147 [21:45<00:00,  8.88s/it]


📊 RAG + Fine-tune 评测结果
  综合得分: 48.0%  (FT only: 67.0%)
    A_simple                 : 60.0%  (15条)
    B_cross_file             : 43.3%  (15条)
    C_fuzzy                  : 8.3%  (12条)
    D_edge                   : 31.2%  (8条)
    auto_generated           : 53.1%  (97条)
✅ 已保存


In [9]:
# Cell D: All — Qwen-1.5B + PE + RAG
from tqdm import tqdm
from collections import defaultdict

print(f"🚀 All（PE+RAG+FT）评测（{len(cases)} 条）...")

all_results = []
total_score = 0

for case in tqdm(cases, desc="All"):
    try:
        candidates = semantic_retrieve(case["query"], k=10)
        reranked = rerank(case["query"], candidates, top_k=5)
        context = build_rag_prompt(case["query"], reranked)
        answer = ft_generate(case["query"], use_system_prompt=True, context=context)
        score = auto_score(answer, case)
        total_score += score
        all_results.append({
            "id": case.get("id"), "category": case.get("category"),
            "query": case["query"],
            "expected_file": case.get("expected_file"),
            "expected_function": case.get("expected_function"),
            "score": score, "response": answer
        })
    except Exception as e:
        all_results.append({"id": case.get("id"), "category": case.get("category"),
                             "query": case["query"], "score": 0, "error": str(e)})

n = len(cases)
cat_scores = defaultdict(lambda: {"score": 0, "count": 0})
for r in all_results:
    cat = r.get("category","unknown")
    cat_scores[cat]["score"] += r.get("score", 0)
    cat_scores[cat]["count"] += 1

final_pct = total_score / (n*2) * 100
print("\n" + "="*55)
print("📊 All（PE+RAG+FT）评测结果")
print("="*55)
print(f"  综合得分: {final_pct:.1f}%")
for cat, v in sorted(cat_scores.items()):
    print(f"    {cat:25s}: {v['score']/(v['count']*2)*100:.1f}%  ({v['count']}条)")
print("="*55)

with open(DRIVE_DIR + "all_results.json", "w", encoding="utf-8") as f:
    json.dump({"summary": {"percentage": round(final_pct,1),
               "by_category": {k: round(v["score"]/(v["count"]*2)*100,1) for k,v in cat_scores.items()}},
               "details": all_results}, f, ensure_ascii=False, indent=2)
print(f"✅ 已保存")

🚀 All（PE+RAG+FT）评测（147 条）...


All: 100%|██████████| 147/147 [07:45<00:00,  3.17s/it]


📊 All（PE+RAG+FT）评测结果
  综合得分: 77.2%
    A_simple                 : 73.3%  (15条)
    B_cross_file             : 80.0%  (15条)
    C_fuzzy                  : 54.2%  (12条)
    D_edge                   : 56.2%  (8条)
    auto_generated           : 82.0%  (97条)
✅ 已保存


In [10]:
# Cell E: 完整消融实验汇总
import json
DRIVE_DIR = "/content/drive/MyDrive/repomind_rag/"

results = {
    "①  基线 DeepSeek（无PE/RAG）":     {"pct": 68.7, "model": "DeepSeek-671B"},
    "②  PE only":                        {"pct": 80.3, "model": "DeepSeek-671B"},
    "③  RAG only":                       {"pct": 94.2, "model": "DeepSeek-671B"},
    "④  PE + RAG":                       {"pct": 93.9, "model": "DeepSeek-671B"},
    "⑤  基线 Qwen-1.5B（无PE/RAG/FT）": {"pct": 26.2, "model": "Qwen-1.5B"},
    "⑥  Fine-tune only":                 {"pct": 67.0, "model": "Qwen-1.5B+LoRA"},
    "⑦  PE + Fine-tune":                 {"pct": 66.7, "model": "Qwen-1.5B+LoRA"},
    "⑧  RAG + Fine-tune":               {"pct": 48.0, "model": "Qwen-1.5B+LoRA"},
    "⑨  All（PE+RAG+FT）":              {"pct": 77.2, "model": "Qwen-1.5B+LoRA"},
}

print("=" * 65)
print("📊 完整消融实验矩阵")
print("=" * 65)
print(f"{'配置':<32} {'模型':<18} {'综合得分':>8}  {'vs DeepSeek基线':>12}")
print("-" * 65)
baseline = 68.7
for name, v in results.items():
    delta = v['pct'] - baseline
    bar = "▲" if delta > 0 else "▼"
    print(f"{name:<32} {v['model']:<18} {v['pct']:>7.1f}%  {bar}{abs(delta):>+.1f}%")
print("=" * 65)

# 各类别完整对比
categories = ["A_simple", "B_cross_file", "C_fuzzy", "D_edge", "auto_generated"]
cat_data = {
    "①基线":    [90.0, 100.0, 83.3, 87.5, 57.2],
    "②PE":      [96.7,  96.7, 95.8,100.0, 71.6],
    "③RAG":     [90.0, 100.0, 83.3, 81.2, 96.4],
    "④PE+RAG":  [86.7, 100.0, 87.5,100.0, 94.3],
    "⑥FT":      [83.3,  76.7, 75.0, 50.0, 63.4],
    "⑦PE+FT":   [83.3,  76.7, 75.0, 50.0, 62.9],
    "⑧RAG+FT":  [60.0,  43.3,  8.3, 31.2, 53.1],
    "⑨All":     [73.3,  80.0, 54.2, 56.2, 82.0],
}

print(f"\n{'类别':<18}", end="")
for cfg in cat_data:
    print(f"{cfg:>9}", end="")
print()
print("-" * 90)
for i, cat in enumerate(categories):
    print(f"{cat:<18}", end="")
    for cfg, vals in cat_data.items():
        print(f"{vals[i]:>8.1f}%", end="")
    print()
print("=" * 90)

# 保存汇总
with open(DRIVE_DIR + "ablation_summary.json", "w", encoding="utf-8") as f:
    json.dump({"results": results, "by_category": cat_data}, f, ensure_ascii=False, indent=2)
print(f"\n✅ 消融汇总已保存: {DRIVE_DIR}ablation_summary.json")

📊 完整消融实验矩阵
配置                               模型                     综合得分  vs DeepSeek基线
-----------------------------------------------------------------
①  基线 DeepSeek（无PE/RAG）          DeepSeek-671B         68.7%  ▼+0.0%
②  PE only                       DeepSeek-671B         80.3%  ▲+11.6%
③  RAG only                      DeepSeek-671B         94.2%  ▲+25.5%
④  PE + RAG                      DeepSeek-671B         93.9%  ▲+25.2%
⑤  基线 Qwen-1.5B（无PE/RAG/FT）      Qwen-1.5B             26.2%  ▼+42.5%
⑥  Fine-tune only                Qwen-1.5B+LoRA        67.0%  ▼+1.7%
⑦  PE + Fine-tune                Qwen-1.5B+LoRA        66.7%  ▼+2.0%
⑧  RAG + Fine-tune               Qwen-1.5B+LoRA        48.0%  ▼+20.7%
⑨  All（PE+RAG+FT）                Qwen-1.5B+LoRA        77.2%  ▲+8.5%

类别                      ①基线      ②PE     ③RAG  ④PE+RAG      ⑥FT   ⑦PE+FT  ⑧RAG+FT     ⑨All
------------------------------------------------------------------------------------------
A_simple              90.0%    96.7%  